# Week 10 Practical — Decision Trees, Bagging, Random Forests & XGBoost

## Learning outcomes
1. Explain roots, splits, branches and leaves; trace predictions.
2. Calculate entropy and information gain manually, including three-way categorical splits.
3. Find thresholds for continuous features; control recursive growth and overfitting.
4. Fit and interpret classification and regression trees.
5. Implement bootstrap sampling; distinguish bagging from random forests.
6. Compare single trees, random forests, gradient boosting and XGBoost using validation data.

**Workflow:** Execute setup cells in order. Complete each TODO. Explain *why* the results occur, not just the scores. Never fit a preprocessing step on validation/test data, and reserve the test set for the final comparison.

**Data:** (A) small synthetic cat dataset reproducing the lecture concepts; (B) scikit-learn's built-in Wisconsin breast-cancer benchmark for realistic ensemble experiments; (C) a synthetic numeric regression dataset. No CSV downloads required. Labels in dataset B are supplied by the library and are used only for machine-learning education, not medical decisions.


## Environment and datasets

In [ ]:
# If XGBoost is unavailable, uncomment and run this line once:
# %pip install xgboost
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, ParameterGrid
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, plot_tree, export_text
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report, confusion_matrix, mean_squared_error, mean_absolute_error, r2_score
from sklearn.datasets import load_breast_cancer, make_regression
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print('Environment ready')

In [ ]:
# Small lecture-aligned dataset: categorical inputs, binary class label, numeric weight.
cat_df = pd.DataFrame({
    'ear_shape': ['Pointy','Oval','Oval','Pointy','Oval','Pointy','Floppy','Oval','Floppy','Floppy'],
    'face_shape': ['Round','Not round','Round','Not round','Round','Round','Not round','Round','Round','Round'],
    'whiskers': ['Present','Present','Absent','Present','Present','Absent','Absent','Absent','Absent','Absent'],
    'cat': [1,1,0,0,1,1,0,1,0,0],
    'weight_lbs': [5.2,6.8,7.1,8.3,9.5,10.2,11.0,12.4,8.9,13.1]
})
print('Lecture example:'); display(cat_df)
print('Class counts:', cat_df['cat'].value_counts().to_dict())
# Larger benchmark with numeric features
cancer = load_breast_cancer(as_frame=True)
X_all = cancer.data.copy()
y_all = cancer.target.copy() # 0 = malignant, 1 = benign (library coding)
X_dev, X_test, y_dev, y_test = train_test_split(X_all,y_all,test_size=.20,random_state=RANDOM_STATE,stratify=y_all)
X_train,X_val,y_train,y_val = train_test_split(X_dev,y_dev,test_size=.25,random_state=RANDOM_STATE,stratify=y_dev)
print('Train/validation/test:', X_train.shape, X_val.shape, X_test.shape)
print('Benchmark labels:', dict(enumerate(cancer.target_names)))

### TODO 01 — Inspect the classification problem

**Your task:** Identify input features and the target; compute class proportions for the cat dataset and for training/validation/test benchmark partitions. Explain the difference between categorical and continuous features, and why stratified splitting is useful.

**Deliverables:** Run the code, display requested output/figure, and write 3–5 sentences interpreting what it shows. Replace `# TODO` sections with your implementation; do not use the held-out test set for tuning.

In [ ]:
# TODO: display feature names, cat class distribution, and benchmark class proportions
# Hint: value_counts(normalize=True), X_train.columns


## 1. Classification trees and entropy

### TODO 02 — Entropy from scratch

**Your task:** Implement binary entropy H(p) = −p log₂(p) − (1−p) log₂(1−p). Treat 0 log₂ 0 as zero. Verify values at p=0, 0.25, 0.5, 0.75, 1; graph entropy across [0,1]. Explain pure versus mixed nodes.

**Deliverables:** Run the code, display requested output/figure, and write 3–5 sentences interpreting what it shows. Replace `# TODO` sections with your implementation; do not use the held-out test set for tuning.

In [ ]:
def binary_entropy(p):
    # TODO: accept scalar or NumPy array, handle endpoints safely
    raise NotImplementedError

# TODO: print five checks and plot H(p) against p


### TODO 03 — Information gain for a three-way categorical feature

**Your task:** Using the 10-row cat dataset, calculate parent entropy and weighted child entropy after splitting by ear_shape (Pointy, Floppy, Oval), face_shape, and whiskers. Compute $IG = H(parent) − Σ(n_{child}/n_{parent}) H(child)$. Display a ranked table and name the best root feature. Do not assume binary branches for categorical features.

**Deliverables:** Run the code, display requested output/figure, and write 3–5 sentences interpreting what it shows. Replace `# TODO` sections with your implementation; do not use the held-out test set for tuning.

In [ ]:
def entropy_labels(labels):
    # TODO: calculate Shannon entropy from arbitrary class counts
    raise NotImplementedError

def categorical_information_gain(frame, feature, target='cat'):
    # TODO: group by feature, weight each child entropy by group size
    raise NotImplementedError

# TODO: evaluate ['ear_shape','face_shape','whiskers'] and sort by gain


### TODO 04 — Train and visualize a classification tree

**Your task:** One-hot encode the three categorical cat features (use pandas.get_dummies), fit DecisionTreeClassifier(criterion="entropy", max_depth=3, random_state=42), plot the tree, print its rules with export_text, and manually trace one training example through the root and a leaf. Explain how one-hot encoding changes the representation of the three-valued ear feature (scikit-learn uses binary splits).

**Deliverables:** Run the code, display requested output/figure, and write 3–5 sentences interpreting what it shows. Replace `# TODO` sections with your implementation; do not use the held-out test set for tuning.

In [ ]:
# TODO: X_cat = pd.get_dummies(...), y_cat = ...
# TODO: fit tree, plot_tree(...), export_text(...)
# TODO: choose a row and explain the path to its predicted label


## 2. Recursive splitting, stopping rules and numeric thresholds

### TODO 05 — Overfitting and stopping criteria

**Your task:** Fit classification trees on X_train/y_train at max_depth ∈ {1,2,3,4,6,None}. Measure train and validation accuracy and node/leaf counts. Plot both accuracy curves against depth (label unlimited depth separately). Repeat with min_samples_leaf ∈ {1,3,8,15}. Identify a reasonable setting without consulting test data.

**Deliverables:** Run the code, display requested output/figure, and write 3–5 sentences interpreting what it shows. Replace `# TODO` sections with your implementation; do not use the held-out test set for tuning.

In [ ]:
# TODO: loop over depth settings and collect get_depth(), get_n_leaves(), accuracy
# TODO: graph training vs validation performance
# TODO: repeat for min_samples_leaf and interpret under/overfitting


### TODO 06 — Continuous feature threshold search

**Your task:** For the benchmark feature "mean radius", sort the distinct training values and generate candidate thresholds halfway between adjacent values. For each threshold, calculate the weighted entropy of the two child nodes and the information gain. Display the top five thresholds and plot information gain vs threshold. Compare the best one-feature split to a depth-1 entropy tree fitted on that feature.

**Deliverables:** Run the code, display requested output/figure, and write 3–5 sentences interpreting what it shows. Replace `# TODO` sections with your implementation; do not use the held-out test set for tuning.

In [ ]:
feature = 'mean radius'
values = X_train[feature].to_numpy()
labels = y_train.to_numpy()
# TODO: candidate thresholds = midpoints of sorted unique values
# TODO: calculate left/right entropy and gain for each candidate
# TODO: plot and compare with DecisionTreeClassifier(max_depth=1, criterion='entropy')


### TODO 07 — Gini impurity versus entropy

**Your task:** Fit two depth-3 trees on the benchmark training data: one with criterion="gini", the other with criterion="entropy". Compare root features, node counts, training accuracy and validation accuracy. Define Gini = 1−Σp_k² and explain why the selected splits can differ.

**Deliverables:** Run the code, display requested output/figure, and write 3–5 sentences interpreting what it shows. Replace `# TODO` sections with your implementation; do not use the held-out test set for tuning.

In [ ]:
# TODO: fit two trees and compare attributes .tree_.feature[0], .get_n_leaves()
# TODO: compute and interpret Gini for a 50/50 node and a pure node


## 3. Regression trees and variance reduction

In [ ]:
# Synthetic regression: a nonlinear signal plus noise, with reproducible train/val/test partitions.
rng = np.random.default_rng(RANDOM_STATE)
x_reg = rng.uniform(-3,3,size=600)
y_reg = 2*np.sin(1.7*x_reg) + 0.35*x_reg + rng.normal(0,.45,size=len(x_reg))
X_reg = pd.DataFrame({'sensor_reading':x_reg})
Xr_dev,Xr_test,yr_dev,yr_test = train_test_split(X_reg,y_reg,test_size=.2,random_state=RANDOM_STATE)
Xr_train,Xr_val,yr_train,yr_val = train_test_split(Xr_dev,yr_dev,test_size=.25,random_state=RANDOM_STATE)
print(Xr_train.shape, Xr_val.shape, Xr_test.shape)

### TODO 08 — Variance and mean-squared-error reduction

**Your task:** Write a function computing node variance (mean squared deviation from node mean). For each possible split of the regression training feature, calculate the weighted child variances and variance reduction. Find the best threshold. Explain why regression leaves predict the mean of their training targets under squared error.

**Deliverables:** Run the code, display requested output/figure, and write 3–5 sentences interpreting what it shows. Replace `# TODO` sections with your implementation; do not use the held-out test set for tuning.

In [ ]:
def node_variance(y):
    # TODO: np.mean((y-np.mean(y))**2) for nonempty y
    raise NotImplementedError

# TODO: enumerate threshold midpoints and compare variance reduction


### TODO 09 — Fit a regression tree and study its depth

**Your task:** Fit DecisionTreeRegressor(max_depth=2), (max_depth=5), and an unconstrained tree. Plot predicted curves over a dense grid with training points. Compute validation RMSE and MAE; select depth using validation only. Explain piecewise-constant predictions and why very deep trees may fit noise.

**Deliverables:** Run the code, display requested output/figure, and write 3–5 sentences interpreting what it shows. Replace `# TODO` sections with your implementation; do not use the held-out test set for tuning.

In [ ]:
# TODO: fit three regressors; draw scatter and predicted step functions
# TODO: calculate RMSE = np.sqrt(mean_squared_error(...)) and MAE


## 4. Bootstrap sampling, bagging, random forests

### TODO 10 — Bootstrap sampling from scratch

**Your task:** Draw 1,000 bootstrap samples of size n=10 from the cat dataset, each with replacement. For each sample, count unique original rows and out-of-bag (OOB) rows. Report the average OOB fraction and compare with the theoretical limit (1−1/n)^n ≈ e⁻¹ for large n. Explain why some examples repeat.

**Deliverables:** Run the code, display requested output/figure, and write 3–5 sentences interpreting what it shows. Replace `# TODO` sections with your implementation; do not use the held-out test set for tuning.

In [ ]:
# TODO: use rng.choice(np.arange(len(cat_df)), size=len(cat_df), replace=True)
# TODO: count unique and omitted indices over 1000 repetitions


### TODO 11 — Bagging versus a single tree

**Your task:** Fit a high-variance unrestricted decision tree and a BaggingClassifier using unrestricted decision trees as base estimators on the benchmark training set. Compare train/validation accuracy and the variability of predictions across at least five random seeds. Use current sklearn parameter `estimator=`; if your installed version requires `base_estimator=`, adapt and note the version.

**Deliverables:** Run the code, display requested output/figure, and write 3–5 sentences interpreting what it shows. Replace `# TODO` sections with your implementation; do not use the held-out test set for tuning.

In [ ]:
# TODO: fit DecisionTreeClassifier() and BaggingClassifier(estimator=DecisionTreeClassifier(), n_estimators=100, random_state=42)
# TODO: repeat for seeds 0..4, summarize validation accuracy mean/std


### TODO 12 — Random forests and random feature selection

**Your task:** Fit RandomForestClassifier(n_estimators=150, bootstrap=True, max_features="sqrt", oob_score=True, random_state=42) and a bagged tree model using all features at each split. Compare validation accuracy, OOB accuracy (training-derived), and feature importances. Explain which randomness source bagging uses and which additional source random forests use.

**Deliverables:** Run the code, display requested output/figure, and write 3–5 sentences interpreting what it shows. Replace `# TODO` sections with your implementation; do not use the held-out test set for tuning.

In [ ]:
# TODO: fit random forest and bagged tree; print validation and OOB scores
# TODO: show 10 largest forest.feature_importances_ with a horizontal bar chart


### TODO 13 — Tune forest complexity

**Your task:** Using validation data only, try n_estimators ∈ {50,150}, max_depth ∈ {3,None}, and min_samples_leaf ∈ {1,4}. Present a table ranked by validation balanced accuracy and record your selected hyperparameters. Do not use the held-out test set during selection.

**Deliverables:** Run the code, display requested output/figure, and write 3–5 sentences interpreting what it shows. Replace `# TODO` sections with your implementation; do not use the held-out test set for tuning.

In [ ]:
# TODO: loop over ParameterGrid({'n_estimators':[50,150], 'max_depth':[3,None], 'min_samples_leaf':[1,4]})
# TODO: store metrics and select best by validation balanced_accuracy_score


## 5. Boosting and XGBoost

### TODO 14 — Gradient boosting versus bagging

**Your task:** Fit GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=2, random_state=42). Compare its validation accuracy and balanced accuracy with your bagging and random forest results. Explain the distinction between independently fitted bootstrap trees and sequentially added trees that optimize a loss.

**Deliverables:** Run the code, display requested output/figure, and write 3–5 sentences interpreting what it shows. Replace `# TODO` sections with your implementation; do not use the held-out test set for tuning.

In [ ]:
# TODO: fit GradientBoostingClassifier and compare validation metrics
# TODO: explain whether gradient boosting necessarily reweights misclassified observations (hint: it fits loss gradients)


**XGBoost installation:** XGBoost is not part of TensorFlow or scikit-learn. If `import xgboost` fails, run `%pip install xgboost`, restart the kernel if needed, and rerun setup. We use `tree_method="hist"` for CPU-friendly training. XGBoost uses gradients (and typically Hessians) of the objective to build successive trees; it is not simply AdaBoost-style reweighting of mistakes.

### TODO 15 — Train an XGBoost classifier

**Your task:** Import XGBClassifier from xgboost. Fit a CPU model using n_estimators=100, max_depth=3, learning_rate=0.1, subsample=0.9, colsample_bytree=0.9, objective="binary:logistic", eval_metric="logloss", tree_method="hist", random_state=42. Report train and validation accuracy and balanced accuracy. Explain each parameter briefly.

**Deliverables:** Run the code, display requested output/figure, and write 3–5 sentences interpreting what it shows. Replace `# TODO` sections with your implementation; do not use the held-out test set for tuning.

In [ ]:
try:
    from xgboost import XGBClassifier
    print('XGBoost available')
except ImportError:
    print('Install XGBoost first: %pip install xgboost')
# TODO: construct, fit, predict and evaluate XGBClassifier


### TODO 16 — XGBoost regularization and learning rate

**Your task:** Compare at least four configurations varying max_depth, learning_rate, reg_lambda (L2) and reg_alpha (L1). Keep n_estimators fixed initially, then discuss why learning rate and n_estimators interact. Rank validation scores and discuss overfitting. For a fair comparison, use the same splits and metrics for all models.

**Deliverables:** Run the code, display requested output/figure, and write 3–5 sentences interpreting what it shows. Replace `# TODO` sections with your implementation; do not use the held-out test set for tuning.

In [ ]:
# TODO: compare parameter dictionaries, e.g. depth 2/5, learning_rate .05/.2, reg_lambda 1/10
# TODO: print validation results and pick the best model without using test labels


### TODO 17 — Final model comparison on untouched test data

**Your task:** Choose the best single tree, random forest, gradient boosting and XGBoost configurations using **validation results only**. Refit each selected configuration on combined training+validation data (optional but recommended), evaluate once on the held-out test set, and report accuracy, balanced accuracy and confusion matrices. Explain why validation and test serve different purposes. Do not select the winner by repeatedly adjusting models against test results.

**Deliverables:** Run the code, display requested output/figure, and write 3–5 sentences interpreting what it shows. Replace `# TODO` sections with your implementation; do not use the held-out test set for tuning.

In [ ]:
# TODO: define the previously selected hyperparameters in a dictionary
# TODO: fit each selected estimator on pd.concat([X_train,X_val]) and pd.concat([y_train,y_val])
# TODO: evaluate ONCE on X_test/y_test and show a comparison table


### TODO 18 — Critical reflection and model recommendation

**Your task:** Write a 250–400 word discussion comparing interpretability, computational cost, bias/variance, categorical-feature handling, numeric thresholds, and ensemble performance. Address: (i) why a single tree is unstable; (ii) why random forests reduce correlation between trees; (iii) why boosting can overfit without regularization; (iv) why no model is guaranteed to win on every dataset; and (v) the limitations of using a small benchmark for high-stakes prediction.

**Deliverables:** Run the code, display requested output/figure, and write 3–5 sentences interpreting what it shows. Replace `# TODO` sections with your implementation; do not use the held-out test set for tuning.

In [ ]:
# TODO: write your discussion in a NEW MARKDOWN CELL below this cell.


## Submission checklist
- [ ] All 18 TODOs completed; no `NotImplementedError` remains.
- [ ] Entropy, information gain, and variance-reduction calculations shown.
- [ ] At least five meaningful visualizations (entropy, tree, depth curve, regression steps, feature importance, etc.).
- [ ] Train/validation/test separation respected.
- [ ] Results are reproducible with fixed random seeds.
- [ ] Interpretation included for every experiment.
- [ ] Final comparison and reflection completed.

## References and further reading
- [scikit-learn Decision Trees User Guide](https://scikit-learn.org/stable/modules/tree.html)
- [scikit-learn Tree Plotting API](https://scikit-learn.org/stable/modules/generated/sklearn.tree.plot_tree.html)
- [scikit-learn Ensemble Methods](https://scikit-learn.org/stable/modules/ensemble.html)
- [XGBoost Python API](https://xgboost.readthedocs.io/en/stable/python/python_api.html)

**Instructor note:** The cat dataset is intentionally tiny and pedagogical; benchmark scores may fluctuate with splits. For production work, use cross-validation, domain-specific evaluation, and external validation.
